# KNN 分類：旅遊問卷資料是否參加 Tour

## 教學情境

假設旅行社對 300 位受訪者進行問卷調查，收集：

- `Age`：年齡
- `Income`：收入
- `Tour`：最後是否參加旅行社 Tour
  - `0` = 沒有參加
  - `1` = 有參加

我們希望使用 **K-Nearest Neighbors（KNN，K最近鄰）**，
根據「年齡」與「收入」來預測新客戶是否會參加 Tour。

> 本範例使用模擬資料，目的在於教學與示範 KNN 的完整流程。

## 1. 載入資料

本課程使用 `tour_survey_300.csv`。

資料格式：

| Age | Income | Tour |
|---:|---:|---:|
| 35 | 58000 | 0 |
| 52 | 92000 | 1 |
| 28 | 45000 | 0 |
| ... | ... | ... |

其中：

- `Age`、`Income` 是 **Features（特徵）**
- `Tour` 是 **Target / Label（目標／類別）**

In [ ]:
import pandas as pd

df = pd.read_csv("tour_survey_300.csv")

print("資料筆數：", len(df))
display(df.head(10))

## 2. 查看資料基本資訊

In [ ]:
print(df.info())
print("\n各欄位統計資料：")
display(df.describe())

## 3. 建立 X 與 y

KNN 要學習的是：

```text
Age + Income
       ↓
      KNN
       ↓
Tour = 0 或 1
```

因此：

- `X` = 輸入特徵
- `y` = 要預測的結果

In [ ]:
X = df[["Age", "Income"]]
y = df["Tour"]

print("X：")
display(X.head())

print("y：")
display(y.head())

## 4. 切分 Training Data 與 Test Data

這次不再使用「前 18 筆訓練、最後 4 筆測試」。

因為如果資料只有 22 筆，Test Data 只有 4 筆，
Accuracy 每一筆就代表 **25%**，評估非常不穩定。

現在有 300 筆資料，因此採用：

- 80% → Training Data
- 20% → Test Data

另外使用 `stratify=y`，讓訓練與測試資料中的 0/1 比例較為一致。

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training Data：", len(X_train))
print("Test Data：", len(X_test))

print("\nTraining Tour 分布：")
print(y_train.value_counts(normalize=True))

print("\nTest Tour 分布：")
print(y_test.value_counts(normalize=True))

## 5. 為什麼 KNN 要做標準化？

KNN 是根據「距離」尋找最近的 K 個鄰居。

本例：

- Age 大約是 20～70
- Income 大約是 25,000～150,000

Income 的數值範圍遠大於 Age。

如果不標準化，Income 很容易在距離計算中占據過大的影響。

因此使用：

```text
StandardScaler
```

將不同尺度的特徵轉換到相近的尺度。

### 非常重要

標準化必須：

```text
Training Data
    ↓
fit + transform

Test Data
    ↓
transform
```

**不能拿 Test Data 重新 fit。**

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_std = scaler.fit_transform(X_train)
X_test_std = scaler.transform(X_test)

print("標準化後 Training Data 前5筆：")
print(X_train_std[:5])

## 6. 建立 KNN 模型

先從 `K=5` 開始。

```text
K = 5
```

代表：

> 當我們要預測一位新客戶時，找出距離他最近的 5 位訓練資料，再由這 5 位鄰居進行多數決。

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5)

knn.fit(X_train_std, y_train)

## 7. 使用 Test Data 進行預測

In [ ]:
y_pred = knn.predict(X_test_std)

print("前20筆預測結果：")
print(y_pred[:20])

print("\n前20筆實際結果：")
print(y_test.to_numpy()[:20])

## 8. 計算 Accuracy

Accuracy 的概念：

```text
Accuracy
=
預測正確的筆數
----------------
全部測試資料筆數
```

In [ ]:
from sklearn.metrics import accuracy_score

acc = accuracy_score(y_test, y_pred)

print("K=5")
print("Test Accuracy：", round(acc, 4))

## 9. 不要直接假設 K=5 最好

KNN 的 `K` 是重要的超參數。

我們可以比較：

```text
K = 1
K = 3
K = 5
K = 7
K = 9
```

但是：

> **不能看哪一個 K 在 Test Data 上最高，就直接選它。**

因為 Test Data 應該保留到最後，作為模型最終的客觀評估。

因此接下來使用 **Cross Validation（交叉驗證）** 找最佳 K。

## 10. 使用 Pipeline

把：

```text
StandardScaler
      ↓
KNN
```

放進同一個 Pipeline。

這樣在交叉驗證時，每一個 Fold 都會正確地只使用該 Fold 的 Training Data 來學習標準化參數，避免資料洩漏（Data Leakage）。

In [ ]:
from sklearn.pipeline import Pipeline

model = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier())
])

## 11. 使用 GridSearchCV 尋找最佳 K

我們讓電腦自動測試不同的 K。

這裡使用 5-Fold Cross Validation：

```text
Training Data
      ↓
 ┌────┬────┬────┬────┬────┐
 │ F1 │ F2 │ F3 │ F4 │ F5 │
 └────┴────┴────┴────┴────┘
```

每次使用 4 個 Fold 訓練、1 個 Fold 驗證，
最後計算平均 Accuracy。

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "knn__n_neighbors": [1, 3, 5, 7, 9, 11, 13, 15]
}

grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy"
)

grid.fit(X_train, y_train)

print("最佳 K：", grid.best_params_["knn__n_neighbors"])
print("最佳 Cross Validation Accuracy：",
      round(grid.best_score_, 4))

## 12. 查看不同 K 的交叉驗證結果

In [ ]:
results = pd.DataFrame(grid.cv_results_)

show_cols = [
    "param_knn__n_neighbors",
    "mean_test_score",
    "std_test_score"
]

display(
    results[show_cols]
    .sort_values("param_knn__n_neighbors")
    .rename(columns={
        "param_knn__n_neighbors": "K",
        "mean_test_score": "Mean Accuracy",
        "std_test_score": "Std"
    })
)

## 13. 使用最佳 K 建立最終模型

`GridSearchCV` 已經找到最佳 K。

接下來使用最佳模型預測「從來沒有參與模型選擇」的 Test Data。

In [ ]:
best_model = grid.best_estimator_

y_pred = best_model.predict(X_test)

test_acc = accuracy_score(y_test, y_pred)

print("最佳 K：", grid.best_params_["knn__n_neighbors"])
print("Test Accuracy：", round(test_acc, 4))

## 14. 查看 Confusion Matrix

除了 Accuracy，我們也可以看看：

- True Positive（TP）
- True Negative（TN）
- False Positive（FP）
- False Negative（FN）

這可以幫助我們了解：

> KNN 到底把哪些人預測錯了？

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix：")
print(cm)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["No Tour (0)", "Tour (1)"]
)

disp.plot()

# 15. 本範例的完整學習流程

```text
問卷資料
   ↓
Age、Income
   ↓
X / y
   ↓
Train / Test Split
   ↓
StandardScaler
   ↓
KNN
   ↓
Cross Validation
   ↓
GridSearchCV
   ↓
找到最佳 K
   ↓
Test Data
   ↓
Accuracy / Confusion Matrix
```

## 最重要的觀念

### ① Test Data 不應該拿來挑 K

Test Data 是最後的「期末考」。

### ② StandardScaler 只能用 Training Data fit

```python
scaler.fit_transform(X_train)
scaler.transform(X_test)
```

### ③ K 是超參數

K=5 不代表一定最好。

### ④ 小資料集的 Accuracy 很容易不穩定

原本只有 4 筆 Test Data：

```text
1 筆 = 25%
```

因此 25% 並不能充分代表模型能力。

現在增加到 300 筆：

```text
240 筆 Training
60 筆 Test
```

評估會比原本 18/4 的切法穩定很多。

### ⑤ KNN 的核心

KNN 最重要的概念可以用一句話記住：

> **找出距離新資料最近的 K 個鄰居，再透過多數決決定分類結果。**

在本例中，就是根據：

```text
Age + Income
     ↓
找相似的客戶
     ↓
預測是否參加 Tour
```